# Lesson 3: Agentic Search

In [1]:
# libraries
from dotenv import load_dotenv
import os
from tavily import TavilyClient

# load environment variables from .env file
_ = load_dotenv()

# connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [2]:
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]


'The new Blackwell GPU uses TSMC 4NP node, GDDR7 memory, and features advanced Tensor Core technology for AI. It supports up to 10 trillion parameters for models. Released in 2024, it powers high-performance computing.'

## Regular search

In [3]:
# choose location (try to change to your own city!)

city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

> Note: search was modified to return expected results in the event of an exception. High volumes of student traffic sometimes cause rate limit exceptions.

In [4]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results  


for i in search(query):
    print(i)

returning previous results due to exception reaching ddg.
https://weather.com/weather/today/l/USCA0987:1:US
https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8


In [5]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."
    
    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup


> Note: This produces a long output, you may want to right click and clear the cell output after you look at it briefly to avoid scrolling past it.

In [6]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs

returning previous results due to exception reaching ddg.
Website: https://weather.com/weather/today/l/USCA0987:1:US


<body><div class="appWrapper DaybreakLargeScreen LargeScreen lightTheme twcTheme DaybreakLargeScreen--appWrapper--ZkDop sidebarExperimentEnabled" id="appWrapper"><div class="region-sidebarNavigation regionSidebarNavigation"><div class="removeIfEmpty" id="WxuSidebarNavigation-sidebarNavigation-sidebar-nav-injection"><div class="SidebarNavigation--sidebarWrapper--v+g78"><nav class="SidebarNavigation--sidebar--A5yHr" data-testid="sidebar-navigation"><div class="SidebarNavigation--sidebarHeader--bMAxo"><button class="SidebarNavigation--overlayControlButton--cSFvn openSidebarOverlayButton" tabindex="0" type="button"><span class="Icon--iconWrapper--vSeDL"><svg class="Icon--icon--ySD-o" fill="none" name="menu" viewbox="0 0 48 48" xmlns="http://www.w3.org/2000/svg">
<title>Menu</title>
<path d="M42 13.714c0-.947-.672-1.714-1.5-1.714h-33c-.828 0-1.5.768-1.5 1.714 0 .947.672 1.7

In [7]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)
    
print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/weather/today/l/USCA0987:1:US


Recent Locations Weather Forecasts Radar & Maps News & Media Products & Account Lifestyle Specialty Forecasts San Francisco, CA Weather Today in San Francisco, CA 6:52 am 7:16 pm Hourly Weather - San Francisco, CA Now Sunny 1 pm Sunny 2 pm Sunny 3 pm Sunny Don't Miss Seasonal Hub 10 Day Weather - San Francisco, CA Today Day Sunny. High 81F. NNE winds shifting to W at 10 to 15 mph. Night Clear to partly cloudy. Low 57F. Winds W at 5 to 10 mph. Tue 16 Day Partly cloudy skies. High 78F. Winds W at 10 to 20 mph. Night Partly cloudy skies early will give way to considerable cloudiness and fog after midnight. Low 56F. Winds W at 10 to 15 mph. Wed 17 Day Foggy early, then partly cloudy later in the day. High 77F. Winds W at 10 to 20 mph. Night Partly cloudy skies in the evening, then becoming cloudy overnight. Low 58F. Winds WSW at 10 to 15 mph. Thu 18 Day Considerable cloudiness. Expect mist and reduced visibilities at times. High 

## Agentic Search

In [8]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1757959982, 'localtime': '2025-09-15 11:13'}, 'current': {'last_updated_epoch': 1757959200, 'last_updated': '2025-09-15 11:00', 'temp_c': 18.3, 'temp_f': 64.9, 'is_day': 1, 'condition': {'text': 'Partly Cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 2.7, 'wind_kph': 4.3, 'wind_degree': 17, 'wind_dir': 'NNE', 'pressure_mb': 1013.0, 'pressure_in': 29.92, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 84, 'cloud': 25, 'feelslike_c': 18.3, 'feelslike_f': 64.9, 'windchill_c': 15.6, 'windchill_f': 60.2, 'heatindex_c': 15.6, 'heatindex_f': 60.1, 'dewpoint_c': 14.4, 'dewpoint_f': 57.9, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 5.0, 'gust_mph': 4.0, 'gust_kph': 6.4}}


In [9]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1757959982,
        "localtime": "2025-09-15 11:13"
    },
    "current": {
        "last_updated_epoch": 1757959200,
        "last_updated": "2025-09-15 11:00",
        "temp_c": 18.3,
        "temp_f": 64.9,
        "is_day": 1,
        "condition": {
            "text": "Partly Cloudy",
            "icon": "//cdn.weatherapi.com/weather/64x64/day/116.png",
            "code": 1003
        },
        "wind_mph": 2.7,
        "wind_kph": 4.3,
        "wind_degree": 17,
        "wind_dir": "NNE",
        "pressure_mb": 1013.0,
        "pressure_in": 29.92,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 84,
        "cloud": 25,
        "feelslike_c": 18.3,
        "feelslike_f": 64.9,
        "windchill_c": 15.6,
      

<img src="./google_sample.png" width="800" height="600">